# Optimizing Toric Code Decoding with Deep Q Learning
This notebook contains an implementation and discussion of the results of 1811.12338, which uses deep reinforcement learning to achieve error correction at a nearly optimial level.  Before doing any physics, we need to leave native Python behind:


In [1]:
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque

We now implement the Toric Code, as well as script that generates electric (vertex) errors.  We will not review the Toric Code in any detail beyond what is necessary for the following and refer the unfamiliar reader to sections one and two of the original paper quant-ph/9707021.  Below is the script - we'll explain it below.

In [2]:
class DefectCenteredToricEnv:
    def __init__(self, d=5):
        self.d = d
        self.reset()

    def reset(self):
        self.syndrome = np.zeros((self.d, self.d), dtype=np.float32)
        self.qubit_actions = np.zeros((self.d, self.d, 2), dtype=np.int32)
        return self._get_anyons()

    def _get_anyons(self):
        anyons = np.argwhere(self.syndrome == 1.0)
        return [tuple(a) for a in anyons]

    def sample_errors(self, error_rate=0.04):
        self.reset()
        for x in range(self.d):
            for y in range(self.d):
                for edge in [0, 1]:
                    if random.random() < error_rate:
                        self._toggle_qubit(x, y, edge)
        return self._get_anyons()

    def _toggle_qubit(self, x, y, edge):
        self.qubit_actions[x, y, edge] ^= 1
        self.syndrome[x, y] = 1.0 - self.syndrome[x, y]
        if edge == 0:
            nx, ny = (x + 1) % self.d, y
        else:
            nx, ny = x, (y + 1) % self.d
        self.syndrome[nx, ny] = 1.0 - self.syndrome[nx, ny]

    def get_centered_state(self, anyon_coord):
        ax, ay = anyon_coord
        shift_x = (self.d // 2) - ax
        shift_y = (self.d // 2) - ay

        centered = np.roll(self.syndrome, shift=shift_x, axis=0)
        centered = np.roll(centered, shift=shift_y, axis=1)

        return np.expand_dims(centered, axis=0)

    def step(self, anyon_coord, direction):
        """
        Executes an action strictly on a specified anyon.
        Per the paper: Reward is ALWAYS -1 per move. No logical success/failure mapping.
        """
        vx, vy = anyon_coord

        if direction == 0:   # Move Up
            self._toggle_qubit((vx - 1) % self.d, vy, 1)
        elif direction == 1: # Move Down
            self._toggle_qubit(vx, vy, 1)
        elif direction == 2: # Move Left
            self._toggle_qubit(vx, (vy - 1) % self.d, 0)
        elif direction == 3: # Move Right
            self._toggle_qubit(vx, vy, 0)

        next_anyons = self._get_anyons()
        done = len(next_anyons) == 0

        reward = -1.0

        return next_anyons, reward, done

    def check_logical_status(self):
        """Helper for evaluation only; not exposed to the agent's reward structure."""
        logical_x1 = np.sum(self.qubit_actions[:, 0, 0]) % 2
        logical_x2 = np.sum(self.qubit_actions[0, :, 1]) % 2
        return (logical_x1 != 0) or (logical_x2 != 0)

Let's review what is in the above python code.  It creates a class that will be used for the "defect centered" approach to reinforcement learning that we introduce momentarily.  The init simply initializes the class with our specified code distance of 5 and resets it to an error free state according the reset method below.  get_anyons locates electric errors.  sample_errors flips random qubits at a specified error rate to force us out of the code space.  It uses a toggle_qubit method, which is defined immediately below it.  check_logical_status will inform us of whether an odd number of logical flips have occured and is used to evaluate the error correction.  We will discuss the get_centered_state and step methods once we motivate and introduce the reinforcement learning framework we employ to correct errors.

Our goal is to use reinforcement learning to teach an agent to emulate the error correction scheme introduced in quant-ph/0110143, which finds error syndromes and flips edges of the shortest possible error chain, since those are the most likely errors.  We do so using *deep Q learning*, which we now introduce.

Deep Q learning is a kind of reinforcement learning, which seeks to optimize the policy $\pi(s,a)$ an agent uses to select an action $a$ based on a state $s$ and the agent's perception of a reward for such an action.  It is this reward which is encoded in a function called $Q$.  This is Q learning.  The "deep" in deep Q learning refers to the fact that we represent Q using a neural network.  Since such a network should be informed by the game we make the agent play to learn $\pi$, let's discuss the game (introduced in 1811.12338) first.

A state corresponds to an error syndrome $\mathcal{S}$ of $N_\mathcal{S}$ anyons/defects $\lbrace e_i \rbrace$.  We can pick any defect and declare a plaquette on its worldline to be the center of the state, defining a defect perspective $P_i$.  The agent can move the central defect one plaquette in any of the four directions, accruing a reward of -1 (the constant reward encourages the agent to minimize the number of moves, mimicing the proceudre in quant-ph/0110143).  This continues until all defects are gone.  The Q function is pased on all possible actions given all possible perspectives.  In the above script, the method get_centered_state defines a defect perspective and the method step plays one the game we just outlined given a fixed anyon.

We now implement the NN that approximates Q.  Since the system is translation invariant, all defect positions should be rewarded equally.  We encourage the neural network to learn this by using a convolutional layer at the beginning (note the ciricular padding to ensure it respects the periodic boundary conditions of the problem).  The remaining layers are all dense layers that gradually reduce the dimension of the output space until it is four, which should correspond to left, right, up, and down moves.  Between each layer, we insert a RELU activation.  There is no activation after the final layer.

In [3]:
class CircularPadding2D(nn.Module):
    def __init__(self, padding):
        super().__init__()
        self.padding = padding

    def forward(self, x):
        x = torch.cat([x[:, :, -self.padding:, :], x, x[:, :, :self.padding, :]], dim=2)
        x = torch.cat([x[:, :, :, -self.padding:], x, x[:, :, :, :self.padding]], dim=3)
        return x

class ToricAnyonDQN(nn.Module):
    def __init__(self, d=5):
        super().__init__()
        self.pad = CircularPadding2D(padding=1)
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=512, kernel_size=3, stride=2)
        self.relu = nn.ReLU()

        self.fc1 = nn.Linear(512 * 3 * 3, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 32)
        self.output_layer = nn.Linear(32, 4)

    def forward(self, x):
        x = self.pad(x)
        x = self.relu(self.conv1(x))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.relu(self.fc4(x))
        return self.output_layer(x)

We now implement deep Q learning.  We do so by breaking the data into mini-batches and using an ADAM optimizer.  We used the gamma as in 1811.12338.  We initially aimed to use their hyperparameters, but our agent didn't really learn anything with their learning rate of .001.  Thus, we doubled the learning rate and the mini-batch size, hoping the increase in the latter would offset the increased risk of catastrophic forgetting that comes with higher learning rate.  We also half the learning rate halfway through for training stability.  We set the number of episodes by balancing the desire for the agent to learn something and the fact that we will borrowing GPU and hope to use little enough of it to give ourselves room to iterate.  We use the value of $\epsilon=0.01$ - a parameter that allows the agent to occassionally take an action that it doesn't perceive to be optimal - from 1811.12338 initially, though we lower it near the end of training for stability.  

The actual learning process is an $\epsilon$-Greedy process in which the agent takes moves the anyon a random direction with probability $\epsilon$ and moves anyons according to $Q$ at the current point in training with probability $1-\epsilon$.  As the network updates, we compare its output to the expected value of $Q$: $$Q =  r + \gamma Q_{next}(1-\delta),$$
where $\delta$ is 1 if the agent is done and 0 otherwise and $Q_{next}$ is the next value of $Q$ maximized over the set of perspectives and actions.  As noted before, we optimze with ADAM.

In [4]:
class ReplayBuffer:
    def __init__(self, capacity=20000):
        self.buffer = deque(maxlen=capacity)
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = zip(*batch)
        return (torch.FloatTensor(np.array(state)),
                torch.LongTensor(action),
                torch.FloatTensor(reward),
                torch.FloatTensor(np.array(next_state)),
                torch.FloatTensor(done))
    def __len__(self):
        return len(self.buffer)

import os

def train_paper_dqn(episodes=5000, batch_size=64, lr=2e-3, gamma=0.95):
    env = DefectCenteredToricEnv(d=5)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    policy_net = ToricAnyonDQN(d=5).to(device)
    target_net = ToricAnyonDQN(d=5).to(device)
    target_net.load_state_dict(policy_net.state_dict())

    optimizer = optim.Adam(policy_net.parameters(), lr=lr)
    memory = ReplayBuffer()

    reward_history = deque(maxlen=100)
    epsilon = 0.1

    # --- PERFORMANCE TRACKING FOR CHECKPOINTING ---
    best_rolling_avg = -float('inf')

    # --- REPLAY WARMUP ---
    print("Warming up replay buffer with random experiences...")
    while len(memory) < 5000:

        anyons = env.sample_errors(error_rate=0.04)
        if len(anyons) == 0: continue
        for _ in range(30):
            chosen_anyon = random.choice(anyons)
            state_centered = env.get_centered_state(chosen_anyon)
            direction = random.randint(0, 3)
            next_anyons, reward, done = env.step(chosen_anyon, direction)

            if not done:
                next_state_centered = env.get_centered_state(random.choice(next_anyons))
            else:
                next_state_centered = np.zeros((1, 5, 5), dtype=np.float32)

            memory.push(state_centered, direction, reward, next_state_centered, done)
            anyons = next_anyons
            if done: break

    print(f"Warmup complete. Starting DQN training on {device}...")

    for episode in range(episodes):

        p = 0.04
        # --- LEARNING RATE SCHEDULER HALFWAY DROP ---
        if episode == (episodes // 2):
            new_lr = 1e-4
            for param_group in optimizer.param_groups:
                param_group['lr'] = new_lr
            print(f"\n[Learning Rate Update] Halfway mark reached. Reducing LR to {new_lr}\n")

        # --- EPSILON COOL-DOWN PHASE ---
        if episode == int(episodes * 0.85): # Drop at 85% to give it a solid runway to finish
            epsilon = 0.01
            print(f"\n[Epsilon Cool-Down] Entering final phase at episode {episode}. Dropping epsilon to {epsilon}\n")

        anyons = env.sample_errors(p)
        total_reward = 0

        if len(anyons) == 0:
            continue

        for step in range(30):
            chosen_anyon = random.choice(anyons)
            state_centered = env.get_centered_state(chosen_anyon)

            if random.random() < epsilon:
                direction = random.randint(0, 3)
            else:
                with torch.no_grad():
                    state_t = torch.FloatTensor(state_centered).unsqueeze(0).to(device)
                    direction = policy_net(state_t).argmax().item()

            next_anyons, reward, done = env.step(chosen_anyon, direction)
            total_reward += reward

            if not done:
                next_state_centered = env.get_centered_state(random.choice(next_anyons))
            else:
                next_state_centered = np.zeros((1, 5, 5), dtype=np.float32)

            memory.push(state_centered, direction, reward, next_state_centered, done)
            anyons = next_anyons

            if len(memory) > batch_size:
                states, actions, rewards, next_states, dones = memory.sample(batch_size)
                states, actions, rewards, next_states, dones = states.to(device), actions.to(device), rewards.to(device), next_states.to(device), dones.to(device)

                current_q = policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
                next_q = target_net(next_states).max(1)[0]
                expected_q = rewards + (gamma * next_q * (1 - dones))

                loss = nn.MSELoss()(current_q, expected_q)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            if done:
                break

        reward_history.append(total_reward)

        # --- TARGET METRIC LOGGING & CHECKPOINT SAVING ---
        if (episode + 1) % 200 == 0:
            target_net.load_state_dict(policy_net.state_dict())
            rolling_avg = np.mean(reward_history)

            # Save if this policy is historically the most optimal shortest-path discoverer
            # Only start checkpointing after episode 1500 so it doesn't over-optimize to the easy p=0.03 data
            if episode >= 1500 and rolling_avg > best_rolling_avg:
                best_rolling_avg = rolling_avg
                torch.save(policy_net.state_dict(), 'best_decoder.pt')
                checkpoint_msg = " -> [NEW BEST MODEL SAVED]"
            else:
                checkpoint_msg = ""

            print(f"Episode {episode+1:4d}/{episodes} | Training p: {p:.2f} | 100-Ep Rolling Avg: {rolling_avg:5.1f} | Epsilon: {epsilon:.2f}{checkpoint_msg}")

    print(f"\nTraining Complete. Best encountered training rolling average: {best_rolling_avg:.2f}")
    return policy_net

Now we evaluate our decoder based on the best decoder.

In [5]:
def evaluate_paper_decoder(model, num_trials=200, error_rate=0.04):
    env = DefectCenteredToricEnv(d=5)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval() # Turns off exploration actions entirely for true verification

    successful_corrections = 0
    logical_failures = 0

    for _ in range(num_trials):
        anyons = env.sample_errors(error_rate=error_rate)
        if len(anyons) == 0:
            successful_corrections += 1
            continue

        done = False
        for step in range(25):
            chosen_anyon = random.choice(anyons)
            state_centered = env.get_centered_state(chosen_anyon)

            with torch.no_grad():
                state_t = torch.FloatTensor(state_centered).unsqueeze(0).to(device)
                direction = model(state_t).argmax().item()

            anyons, reward, done = env.step(chosen_anyon, direction)
            if done:
                # Assess homology layout upon completion
                if not env.check_logical_status():
                    successful_corrections += 1
                else:
                    logical_failures += 1
                break

        if not done: # Ran out of steps
            logical_failures += 1

    success_rate = (successful_corrections / num_trials) * 100
    print("\n================ RESULTS ================")
    print(f"Evaluated over {num_trials} trials at physical error rate p = {error_rate}")
    print(f"Successful Corrections: {successful_corrections} ({success_rate:.2f}%)")
    print(f"Logical Failures: {logical_failures} ({100 - success_rate:.2f}%)")
    print("================================================================")

if __name__ == "__main__":
    # 1. Run the new curriculum training loop
    final_model = train_paper_dqn(episodes=5000)

    # Move final_model to CPU before saving its state_dict to avoid issues
    # if the best_decoder.pt was saved on a different device.
    # This is not strictly necessary for the current error, but good practice.
    final_model.to('cpu')

    # 2. Instantiate a fresh network instance to hold the optimal parameters
    best_model = ToricAnyonDQN(d=5)

    # 3. Load the checkpoint file if it exists, otherwise fall back to final weights
    if os.path.exists('best_decoder.pt'):
        print("\nLoading the best checkpointed model parameters for validation...")
        best_model.load_state_dict(torch.load('best_decoder.pt'))
    else:
        print("\nNo checkpoint file found. Evaluating on final iteration weights...")
        best_model.load_state_dict(final_model.state_dict())

    # Move the model to the correct device AFTER loading its state_dict
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    best_model.to(device)

    # 4. Assess performance at the 4% physical error rate
    evaluate_paper_decoder(best_model, num_trials=200, error_rate=0.04)

Warming up replay buffer with random experiences...
Warmup complete. Starting DQN training on cuda...
Episode  200/5000 | Training p: 0.04 | 100-Ep Rolling Avg: -29.8 | Epsilon: 0.10
Episode  400/5000 | Training p: 0.04 | 100-Ep Rolling Avg: -18.6 | Epsilon: 0.10
Episode  600/5000 | Training p: 0.04 | 100-Ep Rolling Avg: -15.3 | Epsilon: 0.10
Episode  800/5000 | Training p: 0.04 | 100-Ep Rolling Avg: -11.8 | Epsilon: 0.10
Episode 1000/5000 | Training p: 0.04 | 100-Ep Rolling Avg: -10.0 | Epsilon: 0.10
Episode 1200/5000 | Training p: 0.04 | 100-Ep Rolling Avg:  -8.7 | Epsilon: 0.10
Episode 1400/5000 | Training p: 0.04 | 100-Ep Rolling Avg:  -8.1 | Epsilon: 0.10
Episode 1600/5000 | Training p: 0.04 | 100-Ep Rolling Avg:  -8.3 | Epsilon: 0.10 -> [NEW BEST MODEL SAVED]
Episode 2000/5000 | Training p: 0.04 | 100-Ep Rolling Avg:  -6.7 | Epsilon: 0.10 -> [NEW BEST MODEL SAVED]
Episode 2200/5000 | Training p: 0.04 | 100-Ep Rolling Avg:  -8.3 | Epsilon: 0.10
Episode 2400/5000 | Training p: 0.04

These results are good!  More training would likely lead to better accuracy, perhaps even for error rates closer to the ~10% decoding threshold such as those explored in the 1811.12338.